# Phase 3: Deploy agents on Amazon Bedrock AgentCore

In this phase, you will transition from local development to production deployment using Amazon Bedrock AgentCore. You will take the sophisticated multi-agent financial advisory system built in Phase 1 & Phase 2 and deploy it to AWS's enterprise-grade agent hosting platform - AgentCore Runtime.

![architecture](./images/architecture.png)


## What You Will Deploy

You will deploy AgentCore's purpose-built infrastructure for running agents at scale while maintaining security, performance, and reliability standards required for enterprise applications.

## Amazon Bedrock AgentCore Runtime

[Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html) provides a secure, serverless, and purpose-built hosting environment for deploying and running AI agents or tools, shortening the time to value from experiments to production-grade agents.

[Deploy AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html)

## Amazon Bedrock AgentCore Observability

[AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) helps you trace, debug, and monitor agent performance in production environments. It offers detailed visualizations of each step in the agent workflow, enabling you to inspect an agent's execution path, audit intermediate outputs, and debug performance bottlenecks and failures.

In [1]:
%%capture
# Install dependencies for AgentCore deployment
!pip install --force-reinstall -U -r requirements.txt --quiet --disable-pip-version-check

In [2]:
# Import AgentCore Runtime deployment tools and utilities
import uuid
from utils import setup_cognito_user_pool, reauthenticate_user
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
from typing import Any, Optional
import urllib.parse
import requests
import json

In [3]:
# Initialize AWS session and get current region
boto_session = Session()
region = boto_session.region_name

In [4]:
import boto3

region = "eu-north-1"
role_name = f"AmazonBedrockAgentCoreSDKRuntime-{region}"
iam = boto3.client('iam')

# Create the role with correct trust policy
trust_policy = '''{
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}'''

try:
    iam.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=trust_policy
    )
    print(f"✓ Role '{role_name}' created")
except iam.exceptions.EntityAlreadyExistsException:
    # Role exists but trust policy may be wrong — update it
    iam.update_assume_role_policy(
        RoleName=role_name,
        PolicyDocument=trust_policy
    )
    print(f"✓ Role '{role_name}' trust policy updated")

# Attach required policies
policies = [
    "arn:aws:iam::aws:policy/AmazonBedrockFullAccess",
    "arn:aws:iam::aws:policy/CloudWatchLogsFullAccess",
    "arn:aws:iam::aws:policy/AmazonEC2ContainerRegistryReadOnly",
]

for policy_arn in policies:
    iam.attach_role_policy(RoleName=role_name, PolicyArn=policy_arn)

print("✓ Policies attached")
print(f"\nWait ~10 seconds for IAM to propagate, then re-run the launch cell.")


✓ Role 'AmazonBedrockAgentCoreSDKRuntime-eu-north-1' created
✓ Policies attached

Wait ~10 seconds for IAM to propagate, then re-run the launch cell.


### Phase 3.1: Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to `AgentCore Runtime`. To do so we need to:

- Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
- Initialize the App in our code with `app = BedrockAgentCoreApp()`
- Decorate the invocation function with the `@app.entrypoint` decorator
- Let AgentCore Runtime control the running of the agent with `app.run()`

In [5]:
%%writefile main.py
# Create AgentCore-compatible deployment file with streaming endpoint and memory integration

from strands import Agent, tool
from strands.models import BedrockModel
from strands.agent.conversation_manager import SummarizingConversationManager

from budget_agent import FinancialReport, budget_agent
from financial_analysis_agent import financial_analysis_agent
from bedrock_agentcore import BedrockAgentCoreApp
from bedrock_agentcore.memory import MemoryClient

from utils import get_guardrail_id
import uuid
import os
import logging
import boto3

# Get the current AWS region dynamically
region = boto3.Session().region_name

# Configure logging for error tracking and debugging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Initialize AgentCore app
app = BedrockAgentCoreApp()

ORCHESTRATOR_PROMPT = """You are a comprehensive financial advisor orchestrator that coordinates between specialized financial agents to provide complete financial guidance. 

Your specialized agents are:
1. **budget_agent**: Handles budgeting, spending analysis, savings recommendations, and expense tracking
2. **financial_analysis_agent_tool**: Handles investment analysis, stock research, portfolio creation, and performance comparisons

Guidelines for using your agents:
- Use **budget_agent** for questions about: budgets, spending habits, expense tracking, savings goals, debt management
- Use **financial_analysis_agent_tool** for questions about: stocks, investments, portfolios, market analysis, investment recommendations
- You can use both agents together for comprehensive financial planning
- Always provide a cohesive summary that combines insights from multiple agents when applicable
- Maintain a helpful, professional tone and include appropriate disclaimers about financial advice

When a user asks a question:
1. Determine which agent(s) are most appropriate
2. Call the relevant agent(s) with focused queries
3. Synthesize the responses into a coherent, comprehensive answer
4. Provide actionable next steps when possible"""

# Add conversation management to maintain context
conversation_manager = SummarizingConversationManager(
    summary_ratio=0.3,  # Summarize 30% of messages when context reduction is needed
    preserve_recent_messages=5,  # Always keep 5 most recent messages
)

# Configure Bedrock model
bedrock_model = BedrockModel(
    model_id="eu.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,  # Deterministic responses for financial advice
    guardrail_id=get_guardrail_id(),
    guardrail_version="DRAFT",
    guardrail_trace="enabled",
)


@tool
def budget_agent_tool(query: str) -> FinancialReport:
    """Generate structured financial reports with budget analysis and recommendations."""
    logger.info(f"budget_agent_tool called with query: {query[:100]}...")
    try:
        structured_response = budget_agent.structured_output(
            output_model=FinancialReport, prompt=query
        )
        logger.info("budget_agent_tool completed successfully")
        return structured_response
    except Exception as e:
        logger.error(f"Error in budget_agent_tool: {e}", exc_info=True)
        # Return a default structured response on error
        return FinancialReport(
            monthly_income=0.0,
            budget_categories=[],
            recommendations=[f"Error generating report: {str(e)}"],
            financial_health_score=1,
        )


@tool
def financial_analysis_agent_tool(query: str) -> str:
    """Handle investment analysis queries including stock research, portfolio creation, and performance comparisons."""
    logger.info(f"financial_analysis_agent_tool called with query: {query[:100]}...")
    try:
        response = financial_analysis_agent(query)
        logger.info("financial_analysis_agent_tool completed successfully")
        return str(response)
    except Exception as e:
        logger.error(f"Error in financial_analysis_agent_tool: {e}", exc_info=True)
        return f"❌ Financial analysis error: {str(e)}"


# Initialize orchestrator agent at module level
logger.info("Initializing orchestrator agent...")
orchestrator_agent = Agent(
    model=bedrock_model,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[budget_agent_tool, financial_analysis_agent_tool],
    conversation_manager=conversation_manager,
)
logger.info("Orchestrator agent initialized successfully")


@app.entrypoint
async def invoke(payload):
    """Your AI agent function with memory integration"""
    user_message = payload["prompt"]
    actor_id = payload.get("actor_id", "default_user")  # User identifier
    session_id = payload.get("session_id", str(uuid.uuid4()))  # Session identifier
    
    # Retrieve relevant memories before processing
    memory_id = os.environ.get('AGENTCORE_MEMORY_ID')
    if memory_id:
        try:
            memory_client = MemoryClient()
            # Query relevant memories for context
            relevant_memories = memory_client.query_records(
                memory_id=memory_id,
                query=user_message,
                namespace=f"finance/user/{actor_id}/facts",
                max_results=5
            )
            
            # Add memory context to system prompt if available
            if relevant_memories:
                memory_context = "\n\nRelevant user information from previous conversations:\n"
                for record in relevant_memories:
                    memory_context += f"- {record.get('content', '')}\n"
                
                # Enhance agent with memory context
                enhanced_prompt = ORCHESTRATOR_PROMPT + memory_context
                orchestrator_agent.system_prompt = enhanced_prompt
        except Exception as e:
            logger.warning(f"Error retrieving memories: {e}")
    
    # Stream response
    full_response = ""
    async for event in orchestrator_agent.stream_async(user_message):
        if "data" in event:
            chunk = event["data"]
            full_response += chunk
            yield chunk
    
    # Store conversation in memory after response
    if memory_id:
        try:
            memory_client.create_event(
                memory_id=memory_id,
                actor_id=actor_id,
                session_id=session_id,
                content=f"User: {user_message}\nAssistant: {full_response}"
            )
        except Exception as e:
            logger.warning(f"Error storing memory: {e}")


if __name__ == "__main__":
    app.run()

Overwriting main.py


## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to AWS standards

### Phase 3.2: Setting up Amazon Cognito for Authentication

AgentCore Runtime requires authentication. We'll use Amazon Cognito to provide JWT tokens for accessing our deployed agent server.

In [6]:
# Set up Amazon Cognito for AgentCore Runtime authentication
print("Setting up Amazon Cognito user pool...")

cognito_config = setup_cognito_user_pool()

print("Cognito setup completed ✓")
print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

Setting up Amazon Cognito user pool...
✓ Credentials stored in AWS Secrets Manager: agentcore-project-credentials-e18fd181

⚠️  IMPORTANT: SAVE THESE CREDENTIALS NOW!
Username: cloudageuser-d6d03339
Password: tDWK@w2x8ogLfiR%
Secret Name: agentcore-project-credentials-e18fd181
Pool ID: eu-north-1_qkxAUifdt
Discovery URL: https://cognito-idp.eu-north-1.amazonaws.com/eu-north-1_qkxAUifdt/.well-known/openid-configuration
Client ID: 724ml4d7u405drm5dj075bkq4s
Bearer Token: eyJraWQiOiJXQ21KWTBnUGRQZERpZ0RMTmp6bUFhelJ1VncrNzNjWTYzZm9SWmVsNEhBPSIsImFsZyI6IlJTMjU2In0.eyJzdWIiOiJmMGJjNjliYy1mMDMxLTcwMmItNzg5OS05MDJhMzRjMDExMjUiLCJpc3MiOiJodHRwczovL2NvZ25pdG8taWRwLmV1LW5vcnRoLTEuYW1hem9uYXdzLmNvbS9ldS1ub3J0aC0xX3FreEFVaWZkdCIsImNsaWVudF9pZCI6IjcyNG1sNGQ3dTQwNWRybTVkajA3NWJrcTRzIiwib3JpZ2luX2p0aSI6IjM1ZDQ3OWY2LTI0OTctNGRmNi1iZDBkLTM5MzY0OWJiNWYyNiIsImV2ZW50X2lkIjoiMDAyYWE4NmItNzRhYy00MzUwLTgyMWMtNDcwMzIxYTMxMDFlIiwidG9rZW5fdXNlIjoiYWNjZXNzIiwic2NvcGUiOiJhd3MuY29nbml0by5zaWduaW4udXNlci5hZG1pbiIsIm

In [7]:
# Configure JWT authorization for AgentCore Runtime
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [cognito_config["client_id"]],
        "discoveryUrl": cognito_config["discovery_url"],
    }
}

### Phase 3.2.1: Configure AgentCore Memory

AgentCore Memory enables your agent to remember context across conversations. We'll configure three memory strategies:

1. **Semantic Strategy**: Extracts and stores factual information (income, expenses, goals)
2. **User Preference Strategy**: Captures user preferences (risk tolerance, investment style)
3. **Summary Strategy**: Creates conversation summaries for context

This allows the financial advisor to provide personalized advice based on previous interactions.

In [8]:
# Import AgentCore Memory Client
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

In [9]:
# Define memory strategies for financial advisor
memory_strategies = [
    {
        StrategyType.SEMANTIC.value: {
            "name": "financial_facts",
            "description": "Extracts and stores user's financial facts (income, expenses, goals)",
            "namespaces": ["finance/user/{actorId}/facts"]
        }
    },
    {
        StrategyType.USER_PREFERENCE.value: {
            "name": "user_preferences",
            "description": "Captures user preferences for investment risk, budget priorities",
            "namespaces": ["finance/user/{actorId}/preferences"]
        }
    },
    {
        StrategyType.SUMMARY.value: {
            "name": "conversation_summary",
            "description": "Summarizes financial advisory conversations",
            "namespaces": ["finance/user/{actorId}/sessions/{sessionId}"]
        }
    }
]

In [10]:
# Initialize Memory Client and create memory resource
# Note that creating memory resources can take several minutes
memory_client = MemoryClient()

print("Creating AgentCore Memory for financial advisor...")
try:
    # Check if memory already exists
    existing_memories = memory_client.list_memories()
    memory_id = None
    
    for mem in existing_memories:
        if mem.get('name') == 'FinancialAdvisorMemory':
            memory_id = mem['id']
            print(f"✓ Using existing memory: {memory_id}")
            break
    
    if not memory_id:
        memory = memory_client.create_memory_and_wait(
            name="FinancialAdvisorMemory",
            strategies=memory_strategies,
            description="Memory for personal finance advisor agent",
            event_expiry_days=90  # Keep financial data for 90 days
        )
        memory_id = memory["id"]
        print(f"✓ Memory created: {memory_id}")
        print(f"  - Semantic strategy: Extracts financial facts")
        print(f"  - User preference strategy: Captures preferences")
        print(f"  - Summary strategy: Creates conversation summaries")
    
except Exception as e:
    print(f"❌ Error setting up memory: {e}")
    print("Continuing without memory integration...")
    memory_id = None

Creating AgentCore Memory for financial advisor...
✓ Memory created: FinancialAdvisorMemory-ELejL17Ps7
  - Semantic strategy: Extracts financial facts
  - User preference strategy: Captures preferences
  - Summary strategy: Creates conversation summaries


### Phase 3.3: Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables, and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as a container and push to ECR using CI/CD pipelines and IaC.

In this tutorial, we will use the Amazon Bedrock AgentCore Python SDK to easily package your artifacts and deploy them to AgentCore Runtime.

### Phase 3.3.1: Configure AgentCore Runtime deployment

First, we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created, and a requirements file. We will also configure the starter kit to auto-create the Amazon ECR repository on launch.

During the configure step, your Dockerfile will be generated based on your application code.

![runtime](./images/runtime_overview.png)

In [11]:
# Configure AgentCore Runtime deployment settings
import boto3

sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']

agentcore_runtime = Runtime()

agent_name = "personal_finance_agent"

print("Configuring AgentCore Runtime...")

# Construct ARN for your pre-created role
execution_role_arn = f"arn:aws:iam::{account_id}:role/AmazonBedrockAgentCoreSDKRuntime-{region}"

response = agentcore_runtime.configure(
    entrypoint="main.py",
    execution_role=execution_role_arn,   
    auto_create_execution_role=False,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    authorizer_configuration=auth_config,
)

print("Configuration completed ✓")

# Note: You may see a platform mismatch warning (linux/amd64 vs linux/arm64).
# This is expected and can be safely ignored. AgentCore Runtime uses ARM64/Graviton
# processors, and AWS will automatically rebuild the image for the correct architecture
# during deployment. All dependencies in this project are compatible with ARM64.

Entrypoint parsed: file=/Users/uzerali/Downloads/Projects/Ai_Agent/StrandsMultiAgent/StrandsMultiAgent/SourceCode/main.py, bedrock_agentcore_name=main
Configuring BedrockAgentCore agent: personal_finance_agent


Configuring AgentCore Runtime...


⚠️  ℹ️  No container engine found (Docker/Finch/Podman not installed)
✅ Default deployment uses CodeBuild (no container engine needed)
💡 Run 'agentcore launch' for cloud-based building and deployment
💡 For local builds, install Docker, Finch, or Podman

Generated Dockerfile: /Users/uzerali/Downloads/Projects/Ai_Agent/StrandsMultiAgent/StrandsMultiAgent/SourceCode/Dockerfile
Generated .dockerignore: /Users/uzerali/Downloads/Projects/Ai_Agent/StrandsMultiAgent/StrandsMultiAgent/SourceCode/.dockerignore
Keeping 'personal_finance_agent' as default agent
Bedrock AgentCore configured: /Users/uzerali/Downloads/Projects/Ai_Agent/StrandsMultiAgent/StrandsMultiAgent/SourceCode/.bedrock_agentcore.yaml


Configuration completed ✓


**Note:** You may see a platform mismatch warning during configuration:

```
⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
```

**This warning can be safely ignored.** AgentCore Runtime uses AWS Graviton (ARM64) processors for better performance and cost efficiency. AWS will automatically rebuild your Docker image for the correct ARM64 architecture during the deployment process. All Python dependencies used in this project are fully compatible with ARM64.

### Phase 3.3.2: Launching agent to AgentCore Runtime

Now that we have a Dockerfile, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

In [12]:
# Deploy agent to AgentCore Runtime (creates ECR repo and runtime)
print("Launching Agent server to AgentCore Runtime...")
print("This may take several minutes...")

launch_result = agentcore_runtime.launch(
    env_vars={
        "OTEL_PYTHON_EXCLUDED_URLS": "/ping,/invocations",
        "AGENTCORE_MEMORY_ID": memory_id if memory_id else ""
    }
)

print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'personal_finance_agent' to account 741116633751 (eu-north-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: personal_finance_agent


Launching Agent server to AgentCore Runtime...
This may take several minutes...
Repository doesn't exist, creating new ECR repository: bedrock-agentcore-personal_finance_agent


✅ ECR repository available: 741116633751.dkr.ecr.eu-north-1.amazonaws.com/bedrock-agentcore-personal_finance_agent
Using execution role from config: arn:aws:iam::741116633751:role/AmazonBedrockAgentCoreSDKRuntime-eu-north-1
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: personal_finance_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-eu-north-1-5d12c2867b
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-eu-north-1-5d12c2867b
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-eu-north-1-5d12c2867b
✓ Role created: arn:aws:iam::741116633751:role/AmazonBedrockAgentCoreSDKCodeBuild-eu-north-1-5d12c2867b
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-eu-north-1-5d12c2867b
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::741116633751:role/AmazonBedrockAgentCo

Launch completed ✓
Agent ARN: arn:aws:bedrock-agentcore:eu-north-1:741116633751:runtime/personal_finance_agent-ZdpSqj3X8h
Agent ID: personal_finance_agent-ZdpSqj3X8h


#### What Just Happened?

The `agentcore_runtime.launch()` command performed several important operations:

1. **Created an Amazon ECR Repository**: A private container registry to store your agent's Docker image
2. **Built the Docker Image**: Packaged your agent code, dependencies, and configuration into a container
3. **Pushed to ECR**: Uploaded the container image to your ECR repository
4. **Created AgentCore Runtime**: Deployed a serverless runtime environment that will execute your agent
5. **Configured Environment Variables**: Set up memory integration and observability settings

Let's verify what was created by querying AWS services:

In [13]:
import boto3
import json

# Initialize AWS clients
ecr_client = boto3.client('ecr', region_name=region)
agentcore_client = boto3.client('bedrock-agentcore-control', region_name=region)

print("=" * 80)
print("DEPLOYED RESOURCES VERIFICATION")
print("=" * 80)

# 1. Verify ECR Repository
print("\n📦 Amazon ECR Repository:")
print("-" * 80)
try:
    # Extract repository name from agent ARN or use a pattern
    # AgentCore typically creates repos with a specific naming pattern
    repos = ecr_client.describe_repositories(maxResults=10)
    
    # Find the most recently created repository (likely our agent)
    if repos['repositories']:
        latest_repo = sorted(repos['repositories'], 
                           key=lambda x: x['createdAt'], 
                           reverse=True)[0]
        
        print(f"Repository Name: {latest_repo['repositoryName']}")
        print(f"Repository URI: {latest_repo['repositoryUri']}")
        print(f"Created: {latest_repo['createdAt']}")
        
        # Get image details
        images = ecr_client.describe_images(
            repositoryName=latest_repo['repositoryName'],
            maxResults=5
        )
        print(f"\nContainer Images: {len(images['imageDetails'])} image(s)")
        for img in images['imageDetails'][:3]:  # Show up to 3 most recent
            tags = img.get('imageTags', ['<untagged>'])
            size_mb = img['imageSizeInBytes'] / (1024 * 1024)
            print(f"  • Tag: {tags[0]}, Size: {size_mb:.1f} MB, Pushed: {img['imagePushedAt']}")
    else:
        print("No ECR repositories found")
except Exception as e:
    print(f"Error retrieving ECR info: {e}")

# 2. Verify AgentCore Runtime
print("\n🤖 AgentCore Runtime:")
print("-" * 80)
try:
    # List all runtimes and find ours
    runtime_response = agentcore_client.list_agent_runtimes(maxResults=10)
    
    # Find our runtime by matching the agent_id from launch_result
    our_runtime = None
    for runtime in runtime_response.get('agentRuntimes', []):
        if runtime['agentRuntimeId'] == launch_result.agent_id:
            our_runtime = runtime
            break
    
    if our_runtime:
        print(f"Runtime Name: {our_runtime.get('agentRuntimeName', 'N/A')}")
        print(f"Runtime ARN: {our_runtime['agentRuntimeArn']}")
        print(f"Runtime ID: {our_runtime['agentRuntimeId']}")
        print(f"Status: {our_runtime.get('status', 'N/A')}")
        print(f"Version: {our_runtime.get('agentRuntimeVersion', 'N/A')}")
        print(f"Last Updated: {our_runtime.get('lastUpdatedAt', 'N/A')}")
        if our_runtime.get('description'):
            print(f"Description: {our_runtime['description']}")
    else:
        print(f"Runtime not found in list. Showing launch details:")
        print(f"  • Agent ARN: {launch_result.agent_arn}")
        print(f"  • Agent ID: {launch_result.agent_id}")
        print(f"\nNote: Runtime may still be initializing. Total runtimes found: {len(runtime_response.get('agentRuntimes', []))}")
            
except Exception as e:
    print(f"Error retrieving AgentCore Runtime info: {e}")
    print(f"\nShowing runtime details from launch:")
    print(f"  • Agent ARN: {launch_result.agent_arn}")
    print(f"  • Agent ID: {launch_result.agent_id}")

print("\n" + "=" * 80)
print("✅ Verification Complete")
print("=" * 80)
print("\nYour agent is now deployed and ready to be invoked!")

DEPLOYED RESOURCES VERIFICATION

📦 Amazon ECR Repository:
--------------------------------------------------------------------------------
Repository Name: bedrock-agentcore-personal_finance_agent
Repository URI: 741116633751.dkr.ecr.eu-north-1.amazonaws.com/bedrock-agentcore-personal_finance_agent
Created: 2026-08-01 13:24:13.756000+05:30

Container Images: 1 image(s)
  • Tag: latest, Size: 409.2 MB, Pushed: 2026-08-01 13:27:24.437000+05:30

🤖 AgentCore Runtime:
--------------------------------------------------------------------------------
Runtime Name: personal_finance_agent
Runtime ARN: arn:aws:bedrock-agentcore:eu-north-1:741116633751:runtime/personal_finance_agent-ZdpSqj3X8h
Runtime ID: personal_finance_agent-ZdpSqj3X8h
Status: READY
Version: 1
Last Updated: 2026-08-01 07:57:29.008848+00:00

✅ Verification Complete

Your agent is now deployed and ready to be invoked!


### Phase 3.4: Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

In [14]:
# Authenticate user and get bearer token for API access
bearer_token = reauthenticate_user(
    client_id=cognito_config["client_id"],
    secret_name=cognito_config["secret_name"]
)

✓ Retrieved credentials from Secrets Manager: agentcore-project-credentials-e18fd181
✓ Successfully reauthenticated user


In [15]:
def invoke_endpoint(
    agent_arn: str,
    payload,
    session_id: str,
    bearer_token: Optional[str],
    region: str = None,
    endpoint_name: str = "DEFAULT",
) -> Any:
    """Invoke agent endpoint using HTTP request with bearer token."""
    if region is None:
        region = boto3.Session().region_name
    escaped_arn = urllib.parse.quote(agent_arn, safe="")
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    }

    try:
        body = json.loads(payload) if isinstance(payload, str) else payload
    except json.JSONDecodeError:
        body = {"payload": payload}

    try:
        response = requests.post(
            url,
            params={"qualifier": endpoint_name},
            headers=headers,
            json=body,
            timeout=100,
            stream=True,
        )
        last_data = False
        for line in response.iter_lines(chunk_size=1):
            if line:
                line = line.decode("utf-8")
                if line.startswith("data: "):
                    last_data = True
                    line = line[6:]
                    line = line.replace('"', "")
                    yield line
                elif line:
                    line = line.replace('"', "")
                    if last_data:
                        yield "\n" + line
                    last_data = False

    except requests.exceptions.RequestException as e:
        print("Failed to invoke agent endpoint: %s", str(e))
        raise

In [16]:
for chunk in invoke_endpoint(
    agent_arn=launch_result.agent_arn,
    payload={
        "prompt": "I make $6000/month and want to start investing $500/month. Help me create a budget and suggest an investment portfolio.",
        "actor_id": "projectuser_123"
    },
    session_id=str(uuid.uuid4()),
    bearer_token=bearer_token,
):
    print(chunk.replace("\\n", "\n"), end="")

<thinking> The user has provided their monthly income and their intention to invest a specific amount each month. This requires both budgeting and investment portfolio suggestions. I will first use the budget_agent to create a budget that accommodates the user's investment goal. Then, I will use the financial_analysis_agent_tool to suggest an investment portfolio. Finally, I will synthesize the results into a cohesive financial plan. </thinking>
Here is your comprehensive financial plan based on your income and investment goals:

### Budget Plan
- **Monthly Income**: $6000
- **Needs**: $3000 (50%)
- **Wants**: $1800 (30%)
- **Savings and Investments**: $1200 (20%)

### Investment Portfolio Suggestions
Based on your monthly investment of $500, here are three diversified portfolios for different risk levels:

#### Conservative Portfolio
- **Johnson & Johnson (JNJ)**: 25% ($125)
- **Procter & Gamble (PG)**: 20% ($100)
- **Coca-Cola (KO)**: 20% ($100)
- **PepsiCo (PEP)**: 20% ($100)
- **Wa

**Phase complete:** You have successfully deployed your multi-agent system to production using Amazon Bedrock AgentCore with memory integration. Your financial assistant now runs in a fully managed, enterprise-grade environment with:

- **AgentCore Runtime**: Auto-scaling, observability, and security built in
- **AgentCore Memory**: Context retention across conversations using semantic, user preference, and summary strategies
- **Production-Ready Architecture**: Guardrails, authentication, and error handling

The memory integration enables your agent to remember user financial information, preferences, and conversation history, providing personalized advice across sessions. You have completed the full journey from development to production deployment of agentic AI applications on AWS.

### Next Steps

You have completed this notebook. To continue to the next part of the project, return to the project instructions and continue with **Phase 4**.

In [17]:
import boto3

# The control plane client
client = boto3.client('bedrock-agentcore-control', region_name='eu-north-1')

# List runtimes
runtimes = client.list_agent_runtimes()
for rt in runtimes.get('agentRuntimeSummaries', []):
    print(rt)


In [18]:
import boto3

client = boto3.client('bedrock-agentcore-control', region_name='eu-north-1')
print([op for op in dir(client) if not op.startswith('_')])


['can_paginate', 'close', 'create_agent_runtime', 'create_agent_runtime_endpoint', 'create_api_key_credential_provider', 'create_browser', 'create_code_interpreter', 'create_gateway', 'create_gateway_target', 'create_memory', 'create_oauth2_credential_provider', 'create_workload_identity', 'delete_agent_runtime', 'delete_agent_runtime_endpoint', 'delete_api_key_credential_provider', 'delete_browser', 'delete_code_interpreter', 'delete_gateway', 'delete_gateway_target', 'delete_memory', 'delete_oauth2_credential_provider', 'delete_workload_identity', 'exceptions', 'generate_presigned_url', 'get_agent_runtime', 'get_agent_runtime_endpoint', 'get_api_key_credential_provider', 'get_browser', 'get_code_interpreter', 'get_gateway', 'get_gateway_target', 'get_memory', 'get_oauth2_credential_provider', 'get_paginator', 'get_token_vault', 'get_waiter', 'get_workload_identity', 'list_agent_runtime_endpoints', 'list_agent_runtime_versions', 'list_agent_runtimes', 'list_api_key_credential_provider

In [19]:
import boto3

client = boto3.client('bedrock-agentcore-control', region_name='eu-north-1')

# List runtimes
runtimes = client.list_agent_runtimes()
print(runtimes)


{'ResponseMetadata': {'RequestId': 'f256ae8e-59f1-4fba-ad48-1351081bdb2e', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sat, 01 Aug 2026 07:58:46 GMT', 'content-type': 'application/json', 'content-length': '322', 'connection': 'keep-alive', 'x-amzn-requestid': 'f256ae8e-59f1-4fba-ad48-1351081bdb2e', 'x-amz-apigw-id': 'BacQhGJNAi0EUcA=', 'x-amzn-trace-id': 'Root=1-6a6da736-09cf342b5576818b0b823514'}, 'RetryAttempts': 0}, 'agentRuntimes': [{'agentRuntimeArn': 'arn:aws:bedrock-agentcore:eu-north-1:741116633751:runtime/personal_finance_agent-ZdpSqj3X8h', 'agentRuntimeId': 'personal_finance_agent-ZdpSqj3X8h', 'agentRuntimeVersion': '1', 'agentRuntimeName': 'personal_finance_agent', 'lastUpdatedAt': datetime.datetime(2026, 8, 1, 7, 57, 29, 8848, tzinfo=tzutc()), 'status': 'READY'}]}


In [20]:
endpoints = client.list_agent_runtime_endpoints(agentRuntimeId='personal_finance_agent-vOSEzq3UrO')
print(endpoints)


{'ResponseMetadata': {'RequestId': '3476af4c-8a45-4abc-bf2f-7a9e86aedfdb', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sat, 01 Aug 2026 07:58:47 GMT', 'content-type': 'application/json', 'content-length': '23', 'connection': 'keep-alive', 'x-amzn-requestid': '3476af4c-8a45-4abc-bf2f-7a9e86aedfdb', 'x-amz-apigw-id': 'BacQwFpmAi0EolA=', 'x-amzn-trace-id': 'Root=1-6a6da737-54cf0a4779f4c2fe624b2461'}, 'RetryAttempts': 0}, 'runtimeEndpoints': []}
